# Source profiling (task 5)

Profiles every source before cleaning: size, columns, coverage, missing values, duplicates, and how the sources match each other. Section 6 turns the findings into the cleaning rules that `src/transform/clean_backfill.py` implements (task 6). The short version is in `docs/task5_profiling.md`.

The outputs describe the data before cleaning. One cell in section 7 also uses task 6's town maps; it was run before the town review in notebook 07 (4,481 unknown town rows), so running it again now shows 3,152.

In [1]:
from pathlib import Path
import pandas as pd
import re

PROJECT_ROOT = Path.cwd().parent
RAW = PROJECT_ROOT / "data" / "raw"
STAGING = PROJECT_ROOT / "data" / "staging"

for f in sorted(RAW.iterdir()):
    print(f.name)

.gitkeep
Ateneo Policy Center Philippine Political Dynasties Dataset (2022 Update).xlsx
ateneo_politicians_raw.parquet
ateneo_provinces_raw.parquet
hf_memberships.parquet
hf_persons.parquet
openhalalan_winners.csv
psa_poverty.pdf
roster_legislators.pdf


## 1. hf_persons

In [56]:
persons = pd.read_parquet(RAW / "hf_persons.parquet")
print(persons.shape)
persons.head()
persons.info()

(45424, 4)
<class 'pandas.DataFrame'>
RangeIndex: 45424 entries, 0 to 45423
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   id           45424 non-null  str  
 1   first_name   45424 non-null  str  
 2   last_name    45424 non-null  str  
 3   name_suffix  1794 non-null   str  
dtypes: str(4)
memory usage: 3.1 MB


In [57]:
print("id unique:", persons["id"].is_unique)
print("duplicate names:", persons.duplicated(subset=["first_name", "last_name", "name_suffix"]).sum())
persons["name_suffix"].value_counts()
persons["last_name"].value_counts().head(15)

id unique: True
duplicate names: 0


last_name
Tan           165
Reyes         165
Garcia        156
Lim           140
Uy            117
De La Cruz    116
Ramos         115
Mendoza       111
Fernandez     109
Villanueva    105
Bautista      101
Flores        100
Dela Cruz      95
Gonzales       93
Santos         88
Name: count, dtype: int64

In [58]:
cruz = persons[persons["last_name"].str.contains("cruz", case=False)]
cruz["last_name"].value_counts()

last_name
De La Cruz       116
Dela Cruz         95
Cruz              85
Vera Cruz          3
Cruzado            3
Cruzat             2
Cruz-Am            2
Sta Cruz           1
Salva Cruz         1
Veracruz           1
Inza Cruz          1
Cruzada            1
Cruza              1
Dela Cruz Jr       1
Escanlar-Cruz      1
Name: count, dtype: int64

In [59]:
print("extra spaces:", (persons["last_name"] != persons["last_name"].str.strip()).sum())
print("all caps:", persons["last_name"].str.isupper().sum())

extra spaces: 0
all caps: 0


In [60]:
suffix_in_last = persons["last_name"].str.contains(r"\s(?:Jr|Sr|II|III|IV)\.?$", regex=True)
print("suffix inside last_name:", suffix_in_last.sum())
persons.loc[suffix_in_last, ["last_name", "name_suffix"]].head(10)


suffix inside last_name: 6


,last_name,name_suffix
9860,Gabuya Jr,NaN
22456,Pineda Jr,NaN
32000,Dela Cruz Jr,NaN
32062,De Manuel Sr,NaN
35868,Lopez Jr,NaN
43472,Rapanut Sr,NaN


In [61]:
persons["name_suffix"].value_counts()

name_suffix
Jr.       1287
Sr.        224
III        135
II          80
V           25
IV          21
I           14
VI           3
VII          1
Jr. IV       1
XVI          1
XIV          1
Jr. II       1
Name: count, dtype: int64

**What it is:** One row per politician. 45,424 rows, 4 columns (`id`, `first_name`, `last_name`, `name_suffix`).

**Quality**
- `id` is unique, so it works as the primary key.
- `first_name` and `last_name` have no missing values, no leading/trailing spaces, and no all-caps entries.
- `name_suffix` is filled for 1,794 rows (about 4%). This is expected, since most people have no suffix.

**Issues found**
- **Surname variants:** the same name is written differently, e.g. "De La Cruz" (116) vs "Dela Cruz" (95), "Vera Cruz" vs "Veracruz", and abbreviations such as "Sta Cruz".
- **Hyphenated surnames** (e.g. "Escanlar-Cruz") may combine two family names.
- **Suffixes inside `last_name`:** 6 rows (e.g. "Lopez Jr", "Rapanut Sr") have the suffix in the last name while `name_suffix` is empty.
- **Odd suffix values:** formatting is consistent ("Jr.", "Sr.", Roman numerals), but "XVI" and "XIV" (1 each) are implausible and "I" (14) is unusual.
- **Zero duplicate full names** across 45,424 people is unusually clean. The source may have merged different people who share a name (to verify using memberships).
- **Common surnames** (Tan 165, Reyes 165, Garcia 156, Lim 140, Uy 117) are shared by unrelated families, so a surname alone is a weak kinship signal.

**Not in this table:** no middle name and no location. Location must come from `hf_memberships`.

## 2. hf_memberships

In [62]:
memberships = pd.read_parquet(RAW / "hf_memberships.parquet")
print(memberships.shape)
memberships.info()

(86234, 8)
<class 'pandas.DataFrame'>
RangeIndex: 86234 entries, 0 to 86233
Data columns (total 8 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   id         86234 non-null  str  
 1   person_id  86234 non-null  str  
 2   party      86234 non-null  str  
 3   region     86234 non-null  str  
 4   province   86234 non-null  str  
 5   locality   58594 non-null  str  
 6   position   86234 non-null  str  
 7   year       86234 non-null  int32
dtypes: int32(1), str(7)
memory usage: 13.0 MB


In [63]:
orphans = ~memberships["person_id"].isin(persons["id"])
print("memberships with no matching person:", orphans.sum())

memberships with no matching person: 0


In [64]:
memberships["year"].value_counts().sort_index()

year
2004    17372
2007    16610
2010    17507
2013    17164
2016    17581
Name: count, dtype: int64

In [65]:
memberships.loc[memberships["locality"].isna(), "position"].value_counts()

position
COUNCILOR                           17678
PROVINCIAL BOARD MEMBER              3745
MAYOR                                2174
VICE MAYOR                           2170
MEMBER, HOUSE OF REPRESENTATIVES     1082
GOVERNOR                              397
VICE GOVERNOR                         394
Name: count, dtype: int64

In [66]:
pd.crosstab(memberships["position"], memberships["locality"].isna())

locality,False,True
position,,
COUNCILOR,47114,17678
GOVERNOR,0,397
MAYOR,5727,2174
"MEMBER, HOUSE OF REPRESENTATIVES",39,1082
PROVINCIAL BOARD MEMBER,2,3745
VICE GOVERNOR,0,394
VICE MAYOR,5712,2170


In [67]:
town_level = memberships[memberships["position"].isin(["MAYOR", "VICE MAYOR", "COUNCILOR"])]
pd.crosstab(town_level["year"], town_level["locality"].isna())

locality,False,True
year,,
2004,16259,0
2007,15483,0
2010,16380,0
2013,201,15842
2016,10230,6180


In [68]:
town_level.loc[town_level["locality"].isna(), "province"].value_counts().head(10)

province
CEBU             1084
BOHOL             931
CAMARINES SUR     735
BATANGAS          679
CAGAYAN           588
ABRA              540
BULACAN           495
PANGASINAN        478
CAVITE            470
ILOILO            446
Name: count, dtype: int64

In [69]:
t2016 = town_level[town_level["year"] == 2016]
share_missing = t2016["locality"].isna().groupby(t2016["province"]).mean()
share_missing[share_missing > 0].sort_index()

province
ABRA                                   1.000000
AGUSAN DEL NORTE                       1.000000
AGUSAN DEL SUR                         1.000000
AKLAN                                  1.000000
ALBAY                                  1.000000
ANTIQUE                                1.000000
APAYAO                                 1.000000
AURORA                                 1.000000
BASILAN                                1.000000
BATAAN                                 1.000000
BATANES                                1.000000
BATANGAS                               1.000000
BENGUET                                1.000000
BILIRAN                                1.000000
BOHOL                                  1.000000
BUKIDNON                               1.000000
BULACAN                                1.000000
CAGAYAN                                1.000000
CAMARINES NORTE                        1.000000
CAMARINES SUR                          1.000000
CAMIGUIN                       

In [70]:
full = share_missing[share_missing == 1].sort_index()
print(len(full), "provinces fully missing, from", full.index[0], "to", full.index[-1])
share_missing[(share_missing > 0) & (share_missing < 1)]

36 provinces fully missing, from ABRA to NCR, CITY OF MANILA, FIRST DISTRICT


province
NCR, FOURTH DISTRICT    0.131579
NCR, SECOND DISTRICT    0.530612
SULTAN KUDARAT          0.008130
Name: locality, dtype: float64

In [71]:
all_2016 = sorted(t2016["province"].unique())
cutoff = full.index[-1]
before_cutoff = [p for p in all_2016 if p <= cutoff]

print(len(before_cutoff), "provinces from A up to the cutoff")
[p for p in before_cutoff if p not in full.index]

52 provinces from A up to the cutoff


['ILOCOS NORTE',
 'ILOCOS SUR',
 'ILOILO',
 'ISABELA',
 'KALINGA',
 'LA UNION',
 'LAGUNA',
 'LANAO DEL NORTE',
 'LANAO DEL SUR',
 'LEYTE',
 'MAGUINDANAO',
 'MARINDUQUE',
 'MASBATE',
 'MISAMIS OCCIDENTAL',
 'MISAMIS ORIENTAL',
 'MOUNTAIN PROVINCE']

In [72]:
full = share_missing[share_missing == 1].sort_index()
print(len(full), "provinces fully missing, from", full.index[0], "to", full.index[-1])
share_missing[(share_missing > 0) & (share_missing < 1)]

36 provinces fully missing, from ABRA to NCR, CITY OF MANILA, FIRST DISTRICT


province
NCR, FOURTH DISTRICT    0.131579
NCR, SECOND DISTRICT    0.530612
SULTAN KUDARAT          0.008130
Name: locality, dtype: float64

In [73]:
print("Last province in the A block:", full.index[-2])

Last province in the A block: IFUGAO


In [74]:
known = memberships.dropna(subset=["locality"])
missing = town_level[town_level["locality"].isna()]

has_history = missing["person_id"].isin(known["person_id"])
print("missing rows:", len(missing))
print("person has a known town in another year:", has_history.sum())

missing rows: 22022
person has a known town in another year: 13924


In [75]:
towns_per_person = known.groupby(["person_id", "province"])["locality"].nunique()

check = missing.merge(
    towns_per_person.rename("n_towns"),
    left_on=["person_id", "province"],
    right_index=True,
    how="left",
)
check["n_towns"].value_counts(dropna=False)

n_towns
1.0    13748
NaN     8194
2.0       79
3.0        1
Name: count, dtype: int64

### Summary: hf_memberships

**What it is:** One row per person, per position, per election. 86,234 rows, 8 columns (`id`, `person_id`, `party`, `region`, `province`, `locality`, `position`, `year`).

**Quality**
- Every `person_id` matches an `id` in `hf_persons` (0 orphans), so the two tables join cleanly.
- 5 elections, evenly covered: 2004 (17,372), 2007 (16,610), 2010 (17,507), 2013 (17,164), 2016 (17,581). No data after 2016.
- About 1,580 mayor rows per election, consistent with the roughly 1,600 Philippine cities and municipalities.

**Missing `locality` (27,640 rows, 32%)**
- **Expected blanks (5,618):** Provincial Board Member (3,745), House Representative (1,082), Governor (397), and Vice Governor (394) cover a province or district, not one town.
- **Real gaps (22,022):** Councilor (17,678), Mayor (2,174), and Vice Mayor (2,170), each about 27% missing.
- **The gaps are concentrated in two years:**

| Year | Town-level rows missing locality |
|---|---|
| 2004, 2007, 2010 | 0% |
| 2013 | 15,842 of 16,043 (about 99%) |
| 2016 | 6,180 of 16,410 (about 38%) |

- **2013:** locality is missing almost everywhere, so the source essentially did not record towns for this election.
- **2016:** 36 provinces are fully missing, in two separate patterns:
  - **An alphabetical block of 35 provinces**, from ABRA to IFUGAO, all 100% missing. The 16 provinces from ILOCOS NORTE to MOUNTAIN PROVINCE are complete, as are the provinces after NCR. This suggests the first part of an alphabetical collection run failed.
  - **NCR districts:** NCR, City of Manila, First District (100% missing), NCR Second District (53%), NCR Fourth District (13%). NCR Third District is complete. This appears to be a separate issue from the alphabetical block.
  - Sultan Kudarat is also slightly affected (0.8%).

**Backfill potential from each person's own history** (same person, same province, town known in another year):

| Group | Rows | Share |
|---|---|---|
| Exactly one known town | 13,748 | 62.4% |
| Two or three known towns (ambiguous) | 80 | 0.4% |
| No known town in that province | 8,194 | 37.2% |

96 of the 8,194 have a known town in a *different* province (they moved, or the province name differs between years). Rows that cannot be filled from history will need another source (OpenHalalan, section 3).

**Other observations**
- 39 House Representatives and 2 Provincial Board Members have a locality (to check).
- `position` values are written in ALL CAPS.
- NCR uses district-style province names, e.g. "NCR, CITY OF MANILA, FIRST DISTRICT".

## 3. openhalalan_winners

In [76]:
winners = pd.read_csv(RAW / "openhalalan_winners.csv")
print(winners.shape)
winners.info()

(157333, 14)
<class 'pandas.DataFrame'>
RangeIndex: 157333 entries, 0 to 157332
Data columns (total 14 columns):
 #   Column              Non-Null Count   Dtype
---  ------              --------------   -----
 0   Last Name           157326 non-null  str  
 1   First Name          157251 non-null  str  
 2   Middle Name         139887 non-null  str  
 3   Middle Name Source  129693 non-null  str  
 4   Title               1140 non-null    str  
 5   Full Name           157333 non-null  str  
 6   Position            157333 non-null  str  
 7   Party               157203 non-null  str  
 8   Year                157333 non-null  int64
 9   Province            157281 non-null  str  
 10  City                143744 non-null  str  
 11  Region              157281 non-null  str  
 12  Sex                 148681 non-null  str  
 13  Sex Source          148681 non-null  str  
dtypes: int64(1), str(13)
memory usage: 31.4 MB


In [77]:
winners["Year"].value_counts().sort_index()

Year
2001    17490
2004    17368
2007    16608
2010    17502
2013    17161
2016    17774
2019    17793
2022    17818
2025    17819
Name: count, dtype: int64

In [78]:
pd.crosstab(winners["Year"], winners["City"].isna())

City,False,True
Year,,
2001,16394,1096
2004,15522,1846
2007,14912,1696
2010,15452,2050
2013,14905,2256
2016,16625,1149
2019,16630,1163
2022,16644,1174
2025,16660,1159


In [79]:
w2013 = winners[winners["Year"] == 2013]
w2013.loc[w2013["City"].isna(), "Position"].value_counts()

Position
COUNCILOR                           905
PROVINCIAL BOARD MEMBER             725
MEMBER, HOUSE OF REPRESENTATIVES    266
MAYOR                               124
VICE MAYOR                           85
VICE GOVERNOR                        78
GOVERNOR                             73
Name: count, dtype: int64

In [80]:
winners["Position"].value_counts()

Position
COUNCILOR                           118203
VICE MAYOR                           14419
MAYOR                                14398
PROVINCIAL BOARD MEMBER               6737
MEMBER, HOUSE OF REPRESENTATIVES      2108
VICE GOVERNOR                          714
GOVERNOR                               702
SENATOR                                 48
PRESIDENT                                2
VICE PRESIDENT                           2
Name: count, dtype: int64

In [81]:
no_province = winners[winners["Province"].isna()]
no_province["Position"].value_counts()

Position
SENATOR           48
PRESIDENT          2
VICE PRESIDENT     2
Name: count, dtype: int64

In [82]:
winners["Middle Name Source"].value_counts()

Middle Name Source
original       63033
comelec_lec    44986
dilg           13954
self-prior      7408
v8.5             312
Name: count, dtype: int64

In [83]:
hf_prov = set(memberships["province"].unique())
oh_prov = set(winners["Province"].dropna().unique())

print("HF provinces:", len(hf_prov), "| OpenHalalan provinces:", len(oh_prov))
print("Only in HF:", sorted(hf_prov - oh_prov))
print("Only in OpenHalalan:", sorted(oh_prov - hf_prov))

HF provinces: 85 | OpenHalalan provinces: 88
Only in HF: ['COMPOSTELA VALLEY', 'NCR, CITY OF MANILA, FIRST DISTRICT', 'NCR, FOURTH DISTRICT', 'NCR, SECOND DISTRICT', 'NCR, THIRD DISTRICT', 'TAWI-TAWI']
Only in OpenHalalan: ['DAVAO DE ORO', 'MAGUINDANAO DEL NORTE', 'MAGUINDANAO DEL SUR', 'NCR FIRST DISTRICT', 'NCR FOURTH DISTRICT', 'NCR SECOND DISTRICT', 'NCR THIRD DISTRICT', 'SPECIAL GEOGRAPHIC AREA', 'TAWI TAWI']


In [84]:
check_prov = ["DAVAO DE ORO", "MAGUINDANAO", "MAGUINDANAO DEL NORTE",
              "MAGUINDANAO DEL SUR", "SPECIAL GEOGRAPHIC AREA"]
winners[winners["Province"].isin(check_prov)].groupby("Province")["Year"].unique()

Province
DAVAO DE ORO               [2001, 2004, 2007, 2010, 2013, 2016, 2019, 202...
MAGUINDANAO                 [2001, 2004, 2007, 2010, 2013, 2016, 2019, 2022]
MAGUINDANAO DEL NORTE                                                 [2025]
MAGUINDANAO DEL SUR                                                   [2025]
SPECIAL GEOGRAPHIC AREA                                               [2025]
Name: Year, dtype: object

In [85]:
hf_towns = set(memberships["locality"].dropna().unique())
oh_towns = set(winners["City"].dropna().unique())

only_hf = sorted(hf_towns - oh_towns)
only_oh = sorted(oh_towns - hf_towns)

print("HF towns:", len(hf_towns), "| OpenHalalan towns:", len(oh_towns))
print("Only in HF:", len(only_hf), "| Only in OpenHalalan:", len(only_oh))
print("Sample only in HF:", only_hf[:20])
print("Sample only in OpenHalalan:", only_oh[:20])

HF towns: 1463 | OpenHalalan towns: 1517
Only in HF: 254 | Only in OpenHalalan: 308
Sample only in HF: ['AL-BARKA', 'ALAMINOS CITY', 'ALFONSO LISTA (POTIA)', 'AMLAN (AYUQUITAN)', 'ANGELES CITY', 'ANINI-Y', 'ASUNCION (SAUG)', 'BACOLOD CITY', 'BACOLOD-KALAWI (BACOLOD GRANDE)', 'BACUNGAN (LEON T. POSTIGO)', 'BACUNGAN (Leon T. Postigo)', 'BAGO CITY', 'BAGUIO CITY', 'BAIS CITY', 'BALAGTAS (BIGAA)', 'BALANGA CITY', 'BALINDONG (WATU)', 'BANNA (ESPIRITU)', 'BASILISA (RIZAL)', 'BATAC CITY']
Sample only in OpenHalalan: ['AL BARKA', 'ALFONSO LISTA', 'ALFONSO LISTA POTIA', 'AMAI MANABILANG', 'AMAI MANABILANG BUMBARAN', 'AMLAN', 'AMLAN AYUQUITAN', 'ANGELES', 'ANINI Y', 'ANTIPOLO', 'ASUNCION', 'ASUNCION SAUG', 'BACOLOD KALAWI', 'BACOLOD KALAWI BACOLOD GRANDE', 'BACUNGAN', 'BACUNGAN LEON T POSTIGO', 'BAGO', 'BAGUIO', 'BAIS', 'BALAGTAS']


In [86]:
def norm_town(name):
    name = name.upper()
    name = re.sub(r"\(.*?\)", "", name)
    name = name.replace("Ñ", "N")                                  # new: Ñ → N
    name = name.replace("-", " ").replace(".", "").replace("'", "")  # new: also removes apostrophes
    name = re.sub(r"\s+", " ", name).strip()                       # moved: clean spaces BEFORE the CITY rules
    name = re.sub(r"^CITY OF ", "", name)
    name = re.sub(r" CITY$", "", name)
    return name

In [87]:
hf_norm = {norm_town(t) for t in hf_towns}
oh_norm = {norm_town(t) for t in oh_towns}

still_hf = sorted(hf_norm - oh_norm)
still_oh = sorted(oh_norm - hf_norm)

print("Before cleaning:", len(only_hf), "|", len(only_oh))
print("After cleaning:", len(still_hf), "|", len(still_oh))
print("Still only in HF:", still_hf[:20])
print("Still only in OpenHalalan:", still_oh[:20])

Before cleaning: 254 | 308
After cleaning: 5 | 120
Still only in HF: ['MAGADALENA', 'PATEROS TAGUIG', 'PRES CARLOS P GARCIA', 'SAM NARCISO', 'TAGUIG PATEROS']
Still only in OpenHalalan: ['ALFONSO LISTA POTIA', 'AMAI MANABILANG', 'AMAI MANABILANG BUMBARAN', 'AMLAN AYUQUITAN', 'ASUNCION SAUG', 'BACOLOD KALAWI BACOLOD GRANDE', 'BACUNGAN LEON T POSTIGO', 'BALAGTAS BIGAA', 'BALINDONG WATU', 'BALIWAG', 'BANAY BANAY', 'BASILISA RIZAL', 'BAYAWAN CITY TULONG', 'BINDOY PAYABON', 'BROOKE S POINT', 'BULALACAO SAN PEDRO', 'CALANASAN BAYAG', 'CORDOVA', 'D V CHIONGBIAN D M MARCOS', 'DARAGA LOCSIN']


In [88]:
winners["town_norm"] = winners["City"].map(norm_town, na_action="ignore")

overlap = winners[winners["Year"].between(2004, 2016)]
affected = overlap[overlap["town_norm"].isin(still_oh)]

print("Overlap-year rows with a leftover name:", len(affected), "of", len(overlap))
print("Distinct leftover names in overlap years:", affected["town_norm"].nunique())

Overlap-year rows with a leftover name: 3875 of 86413
Distinct leftover names in overlap years: 96


### Summary: openhalalan_winners

**What it is:** Election winners. 157,333 rows, 14 columns, covering 9 elections from 2001 to 2025 (about 17,000 rows each). Column names use Title Case with spaces (e.g. `Last Name`), unlike the snake_case names in HF.

**Strengths**
- **Town (`City`) is 91% complete**, including the years HF is missing: 2013 (14,905 of 17,161 rows, 87%) and 2016 (16,625 of 17,774 rows, 94%).
- **Has `Middle Name` (89% complete)**, which HF lacks. It is a useful kinship clue, since the middle name is usually the mother's maiden surname.

**Middle name sources**

| Source | Rows |
|---|---|
| original | 63,033 |
| comelec_lec | 44,986 |
| dilg | 13,954 |
| self-prior | 7,408 |
| v8.5 | 312 |

10,194 middle names have no recorded source. The meaning of `self-prior` and `v8.5` still needs to be confirmed from OpenHalalan's documentation.

**Gaps**
- In 2013, 1,114 town-level rows (905 Councilors, 124 Mayors, 85 Vice Mayors) have no town in this source either.
- 7 missing last names and 82 missing first names.
- The 52 rows with no Province/Region are national positions (48 Senators, 2 Presidents, 2 Vice Presidents). National coverage looks incomplete, so these rows are not useful for town-level analysis.
- `Sex`, `Sex Source`, and `Title` are not yet profiled.

**Matching with HF (2004–2016)**
- **No person ID**, so rows must be matched on name, province, year, and position.
- **Row counts are nearly identical** in shared years (within 5 rows, except 2016: 17,774 vs 17,581). The two sources likely share an origin, so their agreement is not independent confirmation.
- **Position names match HF exactly.** OpenHalalan also has SENATOR, PRESIDENT, VICE PRESIDENT.
- **Provinces:** 6 mappings needed: 4 NCR districts (comma and naming differences), TAWI-TAWI vs TAWI TAWI, and COMPOSTELA VALLEY (HF) vs DAVAO DE ORO (OpenHalalan, used for all years).
- **Towns:** mismatches dropped from 254 / 308 to 5 / 120 after the `norm_town` cleaning function (uppercase, remove parentheses, Ñ → N, remove hyphens, periods, and apostrophes, remove "CITY OF" / " CITY"). In shared years, 3,875 of 86,413 rows (4.5%) still use a leftover name across 96 distinct names. Together with HF's 5, about 100 manual mappings are needed.

**Open questions for the team**
- Use OpenHalalan's extra years (2001, 2019–2025), or limit the analysis to 2004–2016?
- How to handle provinces created after 2016: Maguindanao del Norte and del Sur (appear only in 2025; MAGUINDANAO runs through 2022) and Special Geographic Area (2025)?

## 4. psa_poverty (staging)

In [89]:
poverty = pd.read_csv(STAGING / "psa_poverty.csv")
print(poverty.shape)
poverty.info()

(320, 13)
<class 'pandas.DataFrame'>
RangeIndex: 320 entries, 0 to 319
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   area               320 non-null    str    
 1   area_key           320 non-null    str    
 2   level              320 non-null    str    
 3   region             317 non-null    str    
 4   year               320 non-null    int64  
 5   poverty_incidence  320 non-null    float64
 6   poverty_threshold  320 non-null    float64
 7   cv                 320 non-null    float64
 8   std_error          320 non-null    float64
 9   ci_lower           320 non-null    float64
 10  ci_upper           320 non-null    float64
 11  flags              291 non-null    str    
 12  page               320 non-null    int64  
dtypes: float64(6), int64(2), str(5)
memory usage: 54.6 KB


In [90]:
poverty.head(10)

,area,area_key,level,region,year,poverty_incidence,poverty_threshold,cv,std_error,ci_lower,ci_upper,flags,page
0,PHILIPPINES,PHILIPPINES,national,NaN,2018,12.1,25814.0,1.4,0.2,11.8,12.5,"r1, 1/, 2/, 3/",18
1,PHILIPPINES,PHILIPPINES,national,NaN,2021,13.2,28794.0,1.4,0.2,12.8,13.5,"r1, 1/, 2/, 3/",18
2,PHILIPPINES,PHILIPPINES,national,NaN,2023,10.9,33296.0,1.5,0.2,10.6,11.2,"r1, 1/, 2/, 3/",18
3,National Capital Region (NCR),NATIONAL CAPITAL REGION,region,National Capital Region (NCR),2018,1.4,28682.0,10.4,0.1,1.1,1.7,"1/, 2/, a/",18
4,National Capital Region (NCR),NATIONAL CAPITAL REGION,region,National Capital Region (NCR),2021,2.2,32978.0,6.7,0.1,1.9,2.5,"1/, 2/, a/",18
5,National Capital Region (NCR),NATIONAL CAPITAL REGION,region,National Capital Region (NCR),2023,1.1,37711.0,9.9,0.1,0.9,1.4,"1/, 2/, a/",18
6,1st District,1ST DISTRICT,district,National Capital Region (NCR),2018,1.9,28682.0,30.8,0.6,0.8,3.1,"a/, b/, c/",18
7,1st District,1ST DISTRICT,district,National Capital Region (NCR),2021,1.1,32978.0,32.9,0.4,0.4,1.8,"a/, b/, c/",18
8,1st District,1ST DISTRICT,district,National Capital Region (NCR),2023,1.6,37711.0,29.6,0.5,0.7,2.5,"a/, b/, c/",18
9,2nd District,2ND DISTRICT,district,National Capital Region (NCR),2018,1.4,28682.0,20.1,0.3,0.9,2.0,"2/, a/, b/, c/",18


In [91]:
poverty["year"].value_counts().sort_index()

year
2018    106
2021    106
2023    108
Name: count, dtype: int64

In [92]:
poverty["level"].value_counts()

level
province    251
region       54
district     12
national      3
Name: count, dtype: int64

In [93]:
names_by_year = poverty.groupby("year")["area_key"].apply(set)

print("Only in 2023:", sorted(names_by_year[2023] - names_by_year[2021]))
print("In 2021 but not 2023:", sorted(names_by_year[2021] - names_by_year[2023]))

Only in 2023: ['MAGUINDANAO DEL NORTE', 'MAGUINDANAO DEL SUR']
In 2021 but not 2023: []


In [94]:
poverty[poverty["area_key"].str.contains("MAGUINDANAO")][
    ["area", "level", "year", "poverty_incidence", "flags", "page"]
]

,area,level,year,poverty_incidence,flags,page
306,Maguindanao,province,2018,40.6,"1/, 3/",21
307,Maguindanao,province,2021,29.8,"1/, 3/",21
308,Maguindanao,province,2023,30.4,"1/, 3/",21
309,Maguindanao del Norte,province,2023,28.4,NaN,21
310,Maguindanao del Sur,province,2023,32.1,NaN,21


In [95]:
psa_prov = set(poverty.loc[poverty["level"] == "province", "area_key"])

print("PSA provinces:", len(psa_prov))
print("Only in PSA:", sorted(psa_prov - oh_prov))
print("Only in OpenHalalan:", sorted(oh_prov - psa_prov))

PSA provinces: 85
Only in PSA: ['COTABATO CITY', 'ISABELA CITY']
Only in OpenHalalan: ['NCR FIRST DISTRICT', 'NCR FOURTH DISTRICT', 'NCR SECOND DISTRICT', 'NCR THIRD DISTRICT', 'SPECIAL GEOGRAPHIC AREA']


In [96]:
winners.loc[winners["town_norm"].isin(["COTABATO", "ISABELA"]), ["Province", "town_norm"]].drop_duplicates()

,Province,town_norm
1317,BASILAN,ISABELA
9571,MAGUINDANAO,COTABATO
10998,NEGROS OCCIDENTAL,ISABELA
149211,MAGUINDANAO DEL NORTE,COTABATO


In [97]:
dupes = poverty.duplicated(subset=["area_key", "level", "year"], keep=False)
print("duplicate area-year rows:", dupes.sum())
poverty.loc[dupes, ["area", "area_key", "level", "year", "poverty_incidence", "page"]]

duplicate area-year rows: 0


,area,area_key,level,year,poverty_incidence,page


### Summary: psa_poverty (staging)

**What it is:** PSA poverty statistics, parsed from `psa_poverty.pdf` into a CSV (staging). 320 rows, 13 columns, in long format: one row per area, per level, per year.

**Columns:** `area`, `area_key` (uppercased name for matching), `level`, `region`, `year`, `poverty_incidence`, `poverty_threshold`, `cv`, `std_error`, `ci_lower`, `ci_upper`, `flags` (PDF footnote marks), `page` (source PDF page).

**Quality**
- All number columns were parsed as numbers, not text.
- No duplicate rows: each area appears once per level per year.
- `page` records the source PDF page for every row, so values can be traced back to the original document.
- `region` is missing only for the 3 national rows (expected). `flags` is empty in 29 rows (no footnotes).

**Coverage**
- **Years:** 2018 (106 rows), 2021 (106), 2023 (108).
- **Levels:** province (251), region (54), district (12, the NCR districts), national (3).

**Key limitations**
- **No overlap with HF (2004–2016).** The nearest elections are OpenHalalan's 2019, 2022, and 2025.
- **The smallest level is province.** There is no town-level poverty data, except Isabela City and Cotabato City, which PSA reports separately as if they were provinces.
- **Precision varies.** Some estimates have a high `cv` (e.g. about 30 for NCR 1st District), so they are less reliable.
- The national values (12.1, 13.2, 10.9) appear to match PSA's poverty incidence among **families**, not individuals (to confirm in the PDF).

**Maguindanao in 2023**
- 2023 has three rows: Maguindanao (30.4), Maguindanao del Norte (28.4), and Maguindanao del Sur (32.1).
- The Maguindanao value lies between the two new provinces, so it is likely a combined figure kept for comparison with 2018 and 2021. Using all three rows together would count Maguindanao twice.

**Matching with OpenHalalan provinces**
- PSA's 85 province names match OpenHalalan's exactly, except:
  - **Only in PSA:** COTABATO CITY and ISABELA CITY. In OpenHalalan these are towns: Isabela in BASILAN, and Cotabato in MAGUINDANAO (through 2022) and MAGUINDANAO DEL NORTE (2025).
  - **Only in OpenHalalan:** the 4 NCR districts (PSA labels them `district`, with names like "1ST DISTRICT") and SPECIAL GEOGRAPHIC AREA (no PSA figure).
- A town named ISABELA also exists in NEGROS OCCIDENTAL, so towns must always be matched by province **and** town together.

**To confirm in the PDF**
- What each flag means (footnotes on the PDF pages, e.g. pages 18 and 21).
- Whether the Basilan and Maguindanao figures exclude Isabela City and Cotabato City.
- Whether `poverty_incidence` refers to families or individuals.

**Open question for the team**
- Option A: link poverty to OpenHalalan's 2019, 2022, and 2025 elections (nearest poverty year for each).
- Option B: use poverty only as background context, without year-by-year linking.

## 5. roster_legislators (staging)

In [98]:
roster = pd.read_csv(STAGING / "roster_legislators.csv")
print(roster.shape)
roster.info()

(5118, 15)
<class 'pandas.DataFrame'>
RangeIndex: 5118 entries, 0 to 5117
Data columns (total 15 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   name             5118 non-null   str    
 1   surname          5118 non-null   str    
 2   region_province  5054 non-null   str    
 3   district         4448 non-null   str    
 4   period_clean     5117 non-null   str    
 5   congress_number  3217 non-null   float64
 6   start_year       5115 non-null   float64
 7   end_year         5115 non-null   float64
 8   years_raw        5115 non-null   str    
 9   years_corrected  5118 non-null   bool   
 10  years_suspect    5118 non-null   bool   
 11  notes            387 non-null    str    
 12  is_party_list    5118 non-null   bool   
 13  is_sectoral      5118 non-null   bool   
 14  page             5118 non-null   int64  
dtypes: bool(4), float64(3), int64(1), str(7)
memory usage: 831.9 KB


In [99]:
roster.head(10)

,name,surname,region_province,district,period_clean,congress_number,start_year,end_year,years_raw,years_corrected,years_suspect,notes,is_party_list,is_sectoral,page
0,"ABAD, AURORA B.",ABAD,Batanes,Lone,6th Congress,6.0,1966.0,1969.0,1966-1969,False,False,NaN,False,False,1
1,"ABAD, FLORENCIO B.",ABAD,Batanes,Lone,8th Congress,8.0,1987.0,1992.0,1987-1992,False,False,NaN,False,False,1
2,"ABAD, FLORENCIO B.",ABAD,Batanes,Lone,10th Congress,10.0,1995.0,1998.0,1995-1998,False,False,NaN,False,False,1
3,"ABAD, FLORENCIO B.",ABAD,Batanes,Lone,11th Congress,11.0,1998.0,2001.0,1998-2001,False,False,NaN,False,False,1
4,"ABAD, FLORENCIO B.",ABAD,Batanes,Lone,12th Congress,12.0,2001.0,2004.0,2001-2004,False,False,NaN,False,False,1
5,"ABAD, HENEDINA R.",ABAD,Batanes,Lone,13th Congress,13.0,2004.0,2007.0,2004-2007,False,False,NaN,False,False,1
6,"ABAD, HENEDINA R.",ABAD,Batanes,Lone,15th Congress,15.0,2010.0,2013.0,2010-2013,False,False,NaN,False,False,1
7,"ABAD, HENEDINA R.",ABAD,Batanes,Lone,16th Congress,16.0,2013.0,2016.0,2013-2016,False,False,NaN,False,False,1
8,"ABAD, HENEDINA R.",ABAD,Batanes,Lone,17th Congress,17.0,2016.0,2017.0,2016-2017*,False,False,"*Died on October 8, 2017 during her term of of...",False,False,1
9,"ABAD, JORGE A.",ABAD,Batanes,Lone,2nd Congress,2.0,1949.0,1953.0,1949-1953,False,False,NaN,False,False,1


In [100]:
roster.loc[roster["congress_number"].isna(), "period_clean"].value_counts()

period_clean
Regular Batasang Pambansa               197
Interim Batasang Pambansa               188
Batasang Bayan                          140
National Assembly                       111
1st Commonwealth National Assembly       99
9th Philippine Legislature               99
2nd Commonwealth National Assembly       98
8th Philippine Legislature               96
10th Philippine Legislature              95
6th Philippine Legislature               93
1st Congress of Commonwealth             93
7th Philippine Legislature               92
4th Philippine Legislature               89
5th Philippine Legislature               89
3rd Philippine Legislature               88
2nd Philippine Legislature               85
1st Philippine Legislature               83
First Congress                           24
5th Philipine Legislature                 5
1st Congress Commonwealth                 3
Seventh Congress                          3
13th Congresss                            3
12th congress      

In [101]:
print("years_corrected:", roster["years_corrected"].sum())
print("years_suspect:", roster["years_suspect"].sum())
print("party list:", roster["is_party_list"].sum())
print("sectoral:", roster["is_sectoral"].sum())

years_corrected: 5
years_suspect: 5
party list: 273
sectoral: 80


In [102]:
no_prov = roster[roster["region_province"].isna()]
print("rows with no province:", len(no_prov))
print("of those, party list or sectoral:", (no_prov["is_party_list"] | no_prov["is_sectoral"]).sum())

rows with no province: 64
of those, party list or sectoral: 0


In [103]:
no_prov[["name", "district", "period_clean", "start_year", "notes", "page"]].head(15)

,name,district,period_clean,start_year,notes,page
15,"ABAD SANTOS, VICENTE B.",NaN,Batasang Bayan,1976.0,*Member of the Cabinet (Secretary of Justice).,2
190,"ALBA, MANUEL S.",NaN,Regular Batasang Pambansa,1984.0,*Member of the Cabinet (Office of Budget and M...,9
212,"ALBERTO, JOSE M.",NaN,Batasang Bayan,1976.0,*Regional Representative-at-Large.,10
226,"ALDABA-LIM, ESTEFANIA",NaN,Batasang Bayan,1976.0,*Member of The Cabinet (Secretary of Social Se...,11
239,"ALDEGUER, JOSE M.",NaN,Batasang Bayan,1976.0,*Regional Representative-at-Large.,11
444,"AQUINO, BALTAZAR",NaN,Batasang Bayan,1976.0,*Member of the Cabinet (Secretary of Public Hi...,20
445,"AQUINO, BALTAZAR",NaN,Interim Batasang Pambansa,1978.0,*Member of the Cabinet (Ministry of Public Hig...,21
537,"ASPIRAS, JOSE D.",NaN,Batasang Bayan,1976.0,*Member of the Cabinet (Secretary of Tourism).,24
1132,"CERILLES, VICENTE M.",NaN,Batasang Bayan,1976.0,*Regional Representative-at-Large.,48
1202,"CLAVE, JACOBO C.",NaN,Batasang Bayan,1976.0,*Member of the Cabinet (Presidential Executive...,51


In [104]:
roster.loc[roster["is_party_list"], "region_province"].value_counts().head(10)

region_province
Party-List    273
Name: count, dtype: int64

In [105]:
explained = no_prov["notes"].str.contains("Cabinet|at-Large", case=False, na=False)
print("explained by notes:", explained.sum(), "of", len(no_prov))
no_prov.loc[~explained, ["name", "district", "period_clean", "start_year", "notes", "page"]]

explained by notes: 48 of 64


,name,district,period_clean,start_year,notes,page
1404,"DATUMANONG, SIMEON A.",NaN,Batasang Bayan,1976.0,NaN,59
1479,"DE VEYRA, JAIME C.",Fourth,2nd Philippine Legislature,1910.0,NaN,63
1594,"DIEZ, CLEMENTINO V.",NaN,1st Commonwealth National Assembly,1935.0,*Unseated after losing his election protest. R...,67
1611,"DIMAPORO, IMELDA Q.",NaN,16th Congress,2013.0,NaN,68
1987,"FLORES, FLORENCIO JR. T.",Second,16th Congress,2013.0,NaN,83
1988,"FLORES, FLORENCIO JR. T.",Second,17th Congress,2016.0,NaN,83
2689,"LEVISTE, JOSE ANTONIO C.",NaN,Batasang Bayan,1976.0,NaN,112
2841,"LUKBAN, JUSTO R.",NaN,2nd Philippine Legislature,1910.0,"*Resigned on February 26, 1911 when Representa...",118
2930,"MAKALINTAL, QUERUBE C.",NaN,Interim Batasang Pambansa,1978.0,NaN,122
3110,"MENDOZA, ESTELITO P.",NaN,Interim Batasang Pambansa,1978.0,NaN,129


In [106]:
modern = roster[(roster["start_year"] >= 2001) & ~roster["is_party_list"] & ~roster["is_sectoral"]]
roster_prov = set(modern["region_province"].dropna().str.upper())

print("Roster places (2001+):", len(roster_prov))
print("Only in roster:", sorted(roster_prov - oh_prov))
print("Only in OpenHalalan:", sorted(oh_prov - roster_prov))

Roster places (2001+): 138
Only in roster: ['ANTIPOLO', 'ANTIPOLO CITY', 'BACOLOD CITY', 'BAGUIO CITY', 'BIÑAN CITY', 'CAGAYAN DE ORO CITY', 'CALOOCAN CITY', 'CEBU CITY', 'COMPOSTELA VALLEY', 'COMPOSTELLA VALLEY', 'DAVAO CITY', 'DINAGAT ISLAND', 'DINAGAT ISLAND*', 'ILIGAN CITY', 'ILIGAN CITY*', 'ILOILO CITY', 'LAPU-LAPU CITY', 'LAS PIÑAS', 'LAS PIÑAS CITY', 'LIPA CITY', 'MAGUINDANAO AND COTABATO CITY', 'MAGUINDANAO WITH COTABATO CITY', 'MAKATI', 'MAKATI CITY', 'MALABON / NAVOTAS', 'MALABON CITY', 'MALABON-NAVOTAS', 'MALABON-NAVOTAS CITY', 'MANDALUYONG CITY', 'MANILA', 'MARIKINA CITY', 'MUNTINLUPA CITY', 'NAVOTAS CITY', 'NORTH COTABATO', 'NORTH COTOBATO', 'NUEVA VISCAYA', 'PARAÑAQUE CITY', 'PART- LIST', 'PARTY LIST', 'PASAY CITY', 'PASIG', 'PASIG CITY', 'PATY-LIST', 'QUEZON CITY', 'SAN JOSE DEL MONTE CITY', 'SAN JUAN CITY', 'SHARIFF KABUNSUAN* WITH COTABATO CITY', 'SOUTH COTABATO AND GENERAL SANTOS CITY', 'SOUTH COTOBATO', 'TAGUIG CITY', 'TAGUIG CITY-PATEROS', 'TAGUIG CITY/PATEROS', 'TA

In [107]:
party_typos = modern["region_province"].str.upper().isin(["PART- LIST", "PARTY LIST", "PATY-LIST"])
print("party-list rows missed by the flag:", party_typos.sum())

party-list rows missed by the flag: 14


In [108]:
print("Latest start year:", roster["start_year"].max())
roster["period_clean"].value_counts().tail(10)

Latest start year: 2019.0


period_clean
Regular Batasang Pambasa                1
Third Philippine Legislature            1
9th Congresss                           1
Fourth Philippine Legislature           1
Ninth Philippine Legislature            1
Tenth Philippine Legislature            1
First National Assembly Commonwealth    1
12nd Congress                           1
Interim Batasang Pamabansa              1
1st National Assembly Commonwealth      1
Name: count, dtype: int64

### Summary: roster_legislators (staging)

**What it is:** The historical roster of the House of Representatives (and earlier legislative bodies), parsed from `roster_legislators.pdf` into a CSV (staging). 5,118 rows, 15 columns, one row per legislator per term. It runs from the 1st Philippine Legislature to terms starting in 2019 (18th Congress).

**Columns:** `name`, `surname`, `region_province`, `district`, `period_clean`, `congress_number`, `start_year`, `end_year`, `years_raw`, `years_corrected`, `years_suspect`, `notes`, `is_party_list`, `is_sectoral`, `page` (source PDF page).

**Strengths**
- **Decades of history**, which is valuable for kinship (task 7). Example: the Abad family held the Batanes lone district across generations (Jorge 1949–1953, Aurora 1966–1969, Florencio 1987–2004, Henedina 2004–2017).
- **`notes` (387 rows)** explain deaths, resignations, unseated members, and non-district seats.
- **Parser quality flags:** `years_corrected` (5 rows) and `years_suspect` (5 rows).
- **`page`** traces each row back to the PDF.

**Names**
- Format is "SURNAME, FIRST M.", with a **middle initial only** (OpenHalalan has full middle names).
- The suffix can sit **between the first name and the middle initial**, e.g. "COJUANGCO, EDUARDO JR. M." and "FLORES, FLORENCIO JR. T."
- **Compound surnames** such as "ABAD SANTOS" are separate families from "ABAD", so surnames must be used in full.

**Periods and congress numbers**
- `congress_number` is filled in only 3,217 of 5,118 rows. Most blanks are expected: older bodies without "Nth Congress" numbering (Philippine Legislatures, Commonwealth National Assemblies, Batasang Bayan, Interim and Regular Batasang Pambansa).
- **The parser missed some real Congresses** because of how they were written: words ("First Congress", "Second Congress", "Seventh Congress"), typos ("13th Congresss", "9th Congresss") and lowercase ("12th congress"). "First Congress" is ambiguous (Commonwealth or Republic), so `start_year` is needed to resolve it.
- **`period_clean` has variants of the same body:** typos ("Philipine", "Pamabansa", "Pambasa"), words vs numerals ("Tenth" vs "10th"), and several spellings of the Commonwealth National Assembly.

**Party-list and sectoral**
- 273 rows are flagged `is_party_list`, with "Party-List" as a placeholder in `region_province` (not a real place).
- **14 more party-list rows were missed** by the flag because the placeholder was misspelled ("PARTY LIST", "PART- LIST", "PATY-LIST"). The true count is at least 287.
- 80 rows are flagged `is_sectoral`.

**Missing province (64 rows)**
- **48 are explained by notes:** Cabinet members and Regional Representatives-at-Large, who did not represent a province.
- **16 are unexplained:**
  - **3 modern rows** inside the HF/OpenHalalan years: DIMAPORO, IMELDA Q. (2013) and FLORES, FLORENCIO JR. T. (2013, 2016). These can likely be recovered from OpenHalalan by matching name and year.
  - **5 Marcos-era rows** (Datumanong, Leviste, Makalintal, Mendoza, Ponce Enrile), likely non-district members whose footnote was not captured.
  - **8 older rows (1910–1954)**, likely parsing gaps (e.g. De Veyra has a district but no province).
- `district` is missing in 670 rows (not yet investigated; partly party-list and sectoral).

**Place names (2001 onward, district representatives only)**
- 138 distinct place names, the messiest of all sources. Differences from OpenHalalan:
  - **Cities listed as provinces** (e.g. QUEZON CITY, MANILA, CEBU CITY), since House districts in large cities belong to the city.
  - **"CITY" variants** (MAKATI vs MAKATI CITY).
  - **Footnote asterisks** in names (ILIGAN CITY\*, DINAGAT ISLAND\*).
  - **Typos** (COMPOSTELLA VALLEY, NORTH COTOBATO, NUEVA VISCAYA, ZAMBOAGA SIBUGAY).
  - **Old or alternate names** (COMPOSTELA VALLEY, WESTERN SAMAR, NORTH COTABATO, DINAGAT ISLAND, TAWI-TAWI, SHARIFF KABUNSUAN).
  - **Shared districts** covering two places (TAGUIG-PATEROS, MALABON-NAVOTAS, SOUTH COTABATO AND GENERAL SANTOS CITY, MAGUINDANAO WITH COTABATO CITY), each written several ways.
- Names only in OpenHalalan are naming differences (DAVAO DE ORO, DINAGAT ISLANDS, SAMAR, TAWI TAWI, NCR districts) or 2025-only provinces, consistent with the roster ending in 2019.

## 6. Cleaning rules for task 6 (with status)

Status: **Done** = implemented in `src/transform/clean_backfill.py` · **Changed** = handled differently than planned · **Open** = not implemented (documented limitation or team decision)

### A. General
- **Done:** files in `data/raw` are never edited. Cleaned outputs go to `data/staging` as `*_clean.parquet`, so the parsers' staging files are not overwritten.
- **Done:** OpenHalalan column names converted to snake_case.
- **Done:** OpenHalalan's province names are the standard. Every cleaned table has a `province_std` column using them.
- **Changed:** paths are built from the script's location (`PROJECT_ROOT`), following `parse_roster.py`, instead of reading them from `.env`.
- **Done:** places are always matched on province + town together.

### B. Person names
1. **Changed:** surname particles ("De La Cruz" / "Dela Cruz") are not rewritten. Matching uses compact keys (`last_key`, `first_key`) that drop spaces and punctuation, so both become `DELACRUZ`.
2. **Open:** "Sta"/"Santa" and "Sto"/"Santo" are not unified (`STAMARIA` and `SANTAMARIA` remain different keys).
3. Suffixes:
   - **Done:** HF suffixes moved out of `last_name` (6 rows).
   - **Done:** roster suffixes extracted from the "SURNAME, FIRST JR. M." format.
   - **Done:** standardized to "Jr." / "Sr.". "XIV" and "XVI" flagged as `suffix_suspect` (2 rows).
   - **Open:** the 14 "I" suffixes were not reviewed.
4. **Done:** compound surnames (e.g. "Abad Santos") are kept whole.
5. **Open:** hyphenated surnames are not flagged (team decision for kinship).
6. **Done:** roster names split into `first_name`, `middle_initial`, `name_suffix`, plus match keys.
7. **Open:** OpenHalalan rows with a missing name (7 last, 82 first) are kept but not flagged. They cannot be matched by name.
8. **Done:** `middle_name_source` is kept in the output. No separate reliability flag.

### C. Provinces
1. **Done:** HF → standard via `HF_PROVINCE_MAP` (6 mappings).
2. **Done:** PSA NCR districts → standard via `PSA_AREA_MAP`. Cotabato City and Isabela City marked `is_city` with their province and town (Cotabato City → Maguindanao del Norte from 2023).
3. Roster:
   - **Done (1987 onward):** uppercase, asterisks removed, typos and old names fixed (`ROSTER_PROVINCE_MAP`). Cities resolved to their province through OpenHalalan, with `ROSTER_CITY_MAP` for ambiguous names (Quezon City, Bacolod City, San Juan). Shared districts resolved through one of their cities.
   - **Done:** party-list placeholders blanked, and the `is_party_list` flag fixed for misspelled placeholders (273 → 287).
   - **Changed:** before 1987, `province_std` is set only when the name matches a current province. Historical names stay in `region_province`.
   - **Open:** Kalinga-Apayao (3 rows, a former province) has no single current equivalent.
4. **Open:** the Maguindanao split and Special Geographic Area are kept as reported (team decision on years).

### D. Towns
1. **Done:** `norm_town` applied everywhere (also removes backticks and curly apostrophes).
2. **Done:** manual mappings in `TOWN_MAP` (typos, abbreviations, old-name variants) and `PROVINCE_TOWN_MAP` (Tagoloan → Tagoloan II, Lanao del Sur only).
3. **Done:** old-name mappings suggested by code, then reviewed.
4. **Changed:** mappings are stored as Python dictionaries in `src/transform/mappings.py`, because `.gitignore` blocks all CSV files.

### E. Locality backfilling (HF memberships)
- **Done:** 20,421 of 22,022 gaps filled (92.7%). Every row's method is recorded in `locality_source`.
- **Done:** provincial and district positions labeled `not_applicable` (5,618).
- **Done:** own history, 13,797 rows (99.79% accuracy, 2010 holdout).
- **Changed:** ambiguous cases (more than one possible town) are rejected rather than resolved by a nearest-year rule.
- **Done:** OpenHalalan same-year match, 5,067 (Tier 1) + 590 (Tier 2), about 98% accuracy (2016 test).
- **Done (new):** OpenHalalan cross-year match in reliable years, 967 rows (99.00% / 98.32% accuracy, 2010 holdout).
- **Done (new):** 2013 matches into towns hit by the OpenHalalan shift were undone. 285 remain `shift_revoked`.
- **Done:** 1,316 rows remain `unfilled`.
- **Open:** the 96 rows whose known town was in a different province were not reviewed individually.

### F. Dataset-specific checks
- HF:
  - **Open:** the 39 House Representatives with a locality (partly explained: some are district labels such as Taguig-Pateros).
  - **Open:** possible merged people (one `id` in distant provinces) not checked.
  - **Open (new):** some people are **split into two IDs** (e.g. Lissa / Lissa Marie Streegan). Important for kinship (task 7).
- OpenHalalan:
  - **Done:** `is_national` flags Senator, President, and Vice President, so they can be excluded from town-level analysis.
  - **Done (new):** `town_shift_suspect` flags 2010 and 2013 rows in towns affected by the shift (2,313 and 2,186 rows).
  - **Open:** the 193 extra rows in 2016. `Sex` not profiled (`Title` checked: honorifics, not suffixes).
- PSA:
  - **Done:** `is_combined_area` flags the 2023 Maguindanao row.
  - **Done:** city rows marked `is_city`; the choice of city vs province figure is left to the analysis.
  - **Open:** meaning of the PDF flags, and whether figures refer to families or individuals.
- Roster:
  - **Done:** `congress_number` fixed (3,217 → 3,260) and `period_std` standardized (34 bodies).
  - **Done:** the 3 modern rows with a blank province recovered from OpenHalalan (`ROSTER_MISSING_PROVINCE`).
  - **Open:** the 5 `years_suspect` rows, the 13 older blank provinces, and the `district` labels ("Lone", "Fourth", …) are not standardized.

### G. Open questions for the team
1. Analyze 2004–2016 only, or include OpenHalalan's 2001 and 2019–2025? (Poverty data exists only for 2018, 2021, and 2023.)
2. How should the Maguindanao split be handled?
3. How should hyphenated surnames count in kinship?
4. How should shared districts (e.g. Taguig-Pateros) be assigned?

### H. Ateneo dataset (Done)

Principle for the staging layer: no rows are deleted. Cleaning adds standardized columns and flags, and the original columns are kept. Results and the town review are in `notebooks/07_ateneo_cleaning.ipynb`, and the columns are described in `docs/task6_cleaning.md`.

**Politicians sheet**
1. **Encoding (Done):** repaired "Ã‘" → "Ñ" in first_name, last_name, party, and municipality (`fix_mojibake`).
2. **Provinces (Done):** mapped the 5 non-standard names. Each Metro Manila city moved to its own NCR district (58 rows).
3. **Province groups (Done):** `province_group` added to every cleaned table. The roster's former Kalinga-Apayao rows have no group.
4. **Towns (Done, changed):**
   - Only Mayor, Vice Mayor, and Councilor rows get `town_std`. "---" counts as missing.
   - Standardized with `norm_town`, `TOWN_MAP`, and `PROVINCE_TOWN_MAP` (now keyed by province group). Manila's districts → MANILA.
   - Instead of the parentheses rule, every unknown name was reviewed with code suggestions (similar spelling, shared surnames): 48 OpenHalalan spellings and Caloocan's three spellings → `TOWN_MAP`, 108 Ateneo names → `PROVINCE_TOWN_MAP`, 4 historical towns → `HISTORICAL_TOWNS`. Unconfirmed rows: 1,511 → 20 (`town_verified`).
   - Added: a town label check (shared surnames with the next election) found towns filed under the wrong place in one election (`ATENEO_PLACE_FIXES`, 7 entries) and 4 towns merged into their neighbor in 2001 (`split_doubled_towns`, 40 rows).
5. **Names (Done):** suffixes moved to `name_suffix` (3,184 rows). 0 → O next to a letter and 1 → I before a letter. 91 rows flagged `name_suspect`. `first_key` and `last_key` added.
6. **Duplicates (Done):** 13 rows flagged `is_duplicate`.
7. **Dynasty flag (Done):** `is_fat_dynasty` kept from Ateneo, unchanged.
8. **Link to HF (Done):** 86,229 of 86,234 rows from 2004–2016 linked, with `hf_membership_id`, the merged `person_id`, and HF's backfilled towns (town-level rows without a town: 22,032 → 1,611).
9. **Party (Done):** kept as reported.

**Province sheet**

10. **(Done)** Long format with `province_std`, `province_group`, `year`, and `fat_dynasty_share_pct` (891 rows, 20 blanks). Recounting from the politicians table matches the sheet.

**Outputs:** `ateneo_politicians_clean.parquet` (207,599 rows) and `ateneo_province_shares_clean.parquet` (891 rows) in `data/staging`.

**Open questions for the team**

5. Who integrates Ateneo into the combined people and terms tables? **Answered:** task 7 adds Ateneo's 1988–1998 officials (`notebooks/03_kinship_engine.ipynb`).
6. Main source for shared years: 2001 (OpenHalalan is more complete), 2019 and 2022 (Ateneo is more complete, except mayors).
7. Hyphenated surnames: Ateneo's 2022 flag counts them for both families. Should task 7 do the same?

## 7. ateneo_politicians (raw)

In [2]:
ateneo = pd.read_parquet(RAW / "ateneo_politicians_raw.parquet")
print(ateneo.shape)
ateneo.info()

(207599, 10)
<class 'pandas.DataFrame'>
RangeIndex: 207599 entries, 0 to 207598
Data columns (total 10 columns):
 #   Column                 Non-Null Count   Dtype
---  ------                 --------------   -----
 0   first_name             207598 non-null  str  
 1   last_name              207599 non-null  str  
 2   party                  207591 non-null  str  
 3   region                 207599 non-null  str  
 4   province               207599 non-null  str  
 5   municipality           173872 non-null  str  
 6   position               207599 non-null  str  
 7   year                   207599 non-null  int64
 8   PSGC_Province          207599 non-null  str  
 9   fat_dynasty_indicator  207599 non-null  str  
dtypes: int64(1), str(9)
memory usage: 31.4 MB


In [3]:
pd.crosstab(ateneo["year"], ateneo["position"], margins=True)

position,COUNCILOR,GOVERNOR,MAYOR,"MEMBER, HOUSE OF REPRESENTATIVES",PROVINCIAL BOARD MEMBER,VICE GOVERNOR,VICE MAYOR,All
year,,,,,,,,
1987,0,0,0,193,0,0,0,193
1988,12737,69,1564,0,614,73,1566,16623
1992,12856,73,1575,200,658,74,1576,17012
1995,13033,76,1596,201,673,75,1598,17252
1998,12951,78,1589,208,719,78,1586,17209
2001,12995,79,1585,209,722,77,1585,17252
2004,13078,79,1590,212,744,78,1591,17372
2007,12462,80,1509,218,750,79,1512,16610
2010,13165,78,1618,225,746,78,1597,17507


In [4]:
hf = pd.read_parquet(RAW / "hf_memberships.parquet")
oh = pd.read_csv(RAW / "openhalalan_winners.csv")

pd.DataFrame({
    "ateneo": ateneo["year"].value_counts(),
    "hf": hf["year"].value_counts(),
    "openhalalan": oh["Year"].value_counts(),
}).sort_index().astype("Int64")

,ateneo,hf,openhalalan
1987,193,<NA>,<NA>
1988,16623,<NA>,<NA>
1992,17012,<NA>,<NA>
1995,17252,<NA>,<NA>
1998,17209,<NA>,<NA>
2001,17252,<NA>,17490
2004,17372,17372,17368
2007,16610,16610,16608
2010,17507,17507,17502
2013,17164,17164,17161


In [5]:
TOWN_POSITIONS = ["MAYOR", "VICE MAYOR", "COUNCILOR"]
a_town = ateneo[ateneo["position"].isin(TOWN_POSITIONS)]
h_town = hf[hf["position"].isin(TOWN_POSITIONS)]

pd.DataFrame({
    "ateneo_no_town": a_town["municipality"].isna().groupby(a_town["year"]).sum(),
    "hf_no_town": h_town["locality"].isna().groupby(h_town["year"]).sum(),
}).astype("Int64")

,ateneo_no_town,hf_no_town
year,,
1988,0,<NA>
1992,0,<NA>
1995,0,<NA>
1998,0,<NA>
2001,0,<NA>
2004,0,0
2007,0,0
2010,0,0
2013,15842,15842


In [6]:
persons = pd.read_parquet(RAW / "hf_persons.parquet")
hf_named = hf.merge(persons, left_on="person_id", right_on="id")

for yr in [2004, 2007, 2010, 2013, 2016]:
    a = ateneo.loc[ateneo["year"] == yr, "last_name"].value_counts()
    h = hf_named.loc[hf_named["year"] == yr, "last_name"].str.upper().value_counts()
    diff = a.sub(h, fill_value=0)
    print(yr, "| surnames with a different count:", (diff != 0).sum(), list(diff[diff != 0].index))

2004 | surnames with a different count: 2 ['CARIN0', 'CARINO']
2007 | surnames with a different count: 2 ['CARIN0', 'CARINO']
2010 | surnames with a different count: 4 ['BERMEJ0', 'BERMEJO', 'CARIN0', 'CARINO']
2013 | surnames with a different count: 0 []
2016 | surnames with a different count: 4 ['DAMPIL;AG', 'DAMPILAG', 'SILVERI0', 'SILVERIO']


In [7]:
prov = ateneo.groupby(["province", "PSGC_Province"]).size().rename("rows").reset_index()
print("province names:", prov["province"].nunique(), "| PSGC codes:", prov["PSGC_Province"].nunique(), "| name-code pairs:", len(prov))

std = set(oh["Province"].dropna())
print("Ateneo names not in the standard:", sorted(set(ateneo["province"]) - std))
print("Standard names not in Ateneo:", sorted(std - set(ateneo["province"])))

province names: 85 | PSGC codes: 85 | name-code pairs: 85
Ateneo names not in the standard: ['NCR - FOURTH DISTRICT', 'NCR - MANILA', 'NCR - SECOND DISTRICT', 'NCR - THIRD DISTRICT', 'TAWI-TAWI']
Standard names not in Ateneo: ['MAGUINDANAO DEL NORTE', 'MAGUINDANAO DEL SUR', 'NCR FIRST DISTRICT', 'NCR FOURTH DISTRICT', 'NCR SECOND DISTRICT', 'NCR THIRD DISTRICT', 'SPECIAL GEOGRAPHIC AREA', 'TAWI TAWI']


In [8]:
first_year = ateneo.groupby("province")["year"].min()
first_year[first_year > 1988].sort_values()

province
SARANGANI            1992
BILIRAN              1995
GUIMARAS             1995
DAVAO DE ORO         1998
ZAMBOANGA SIBUGAY    2001
DINAGAT ISLANDS      2007
DAVAO OCCIDENTAL     2016
Name: year, dtype: int64

In [9]:
ipil = ateneo[ateneo["municipality"].eq("IPIL") & ateneo["position"].eq("MAYOR")]
ipil[["year", "province", "municipality", "first_name", "last_name"]].sort_values("year")

,year,province,municipality,first_name,last_name
70842,1988,ZAMBOANGA DEL SUR,IPIL,JOSE,FONTANOZA
130717,1992,ZAMBOANGA DEL SUR,IPIL,FRANCISCO,PONTANAR
130718,1995,ZAMBOANGA DEL SUR,IPIL,FRANCISCO,PONTANAR
118349,1998,ZAMBOANGA DEL SUR,IPIL,REY ANDRE,OLEGARIO
118350,2001,ZAMBOANGA SIBUGAY,IPIL,REY ANDRE,OLEGARIO
118352,2004,ZAMBOANGA SIBUGAY,IPIL,REY ANDRE,OLEGARIO
8245,2007,ZAMBOANGA SIBUGAY,IPIL,ELDWIN,ALIBUTDAN
8248,2010,ZAMBOANGA SIBUGAY,IPIL,ELDWIN,ALIBUTDAN
118366,2016,ZAMBOANGA SIBUGAY,IPIL,ANAMEL,OLEGARIO
180920,2019,ZAMBOANGA SIBUGAY,IPIL,INDAYAMY,OLEGARIO


In [10]:
ncr = ateneo[ateneo["province"].str.startswith("NCR")]
for town in ["PASIG", "PATEROS", "SAN JUAN", "NAVOTAS"]:
    rows = ncr[ncr["municipality"].str.contains(town, na=False)]
    print(town, rows.groupby("province").size().to_dict())

PASIG {'NCR - FOURTH DISTRICT': 42, 'NCR - SECOND DISTRICT': 129}
PATEROS {'NCR - FOURTH DISTRICT': 161, 'NCR - THIRD DISTRICT': 12}
SAN JUAN {'NCR - SECOND DISTRICT': 166, 'NCR - THIRD DISTRICT': 3}
NAVOTAS {'NCR - SECOND DISTRICT': 1, 'NCR - THIRD DISTRICT': 168}


In [11]:
prov_pos = ["GOVERNOR", "VICE GOVERNOR", "PROVINCIAL BOARD MEMBER", "MEMBER, HOUSE OF REPRESENTATIVES"]
p = ateneo[ateneo["position"].isin(prov_pos) & ateneo["municipality"].notna()]
print("provincial positions with a town:", len(p))
print(p["municipality"].value_counts().head(6))
print(p.loc[p["municipality"].str.startswith(("MEMBER,", "PROVINCIAL")), "year"].value_counts())

provincial positions with a town: 1486
municipality
MEMBER, SANGGUNIANG PANLALAWIGAN    782
MEMBER, HOUSE OF REPRESENTATIVES    253
PROVINCIAL GOVERNOR                  81
PROVINCIAL VICE-GOVERNOR             81
PATEROS-TAGUIG                        8
QUEZON CITY                           6
Name: count, dtype: int64
year
2022    1197
Name: count, dtype: int64


In [12]:
broken = ateneo["municipality"].str.contains("Ã", na=False)
print("rows:", broken.sum(), "| distinct names:", ateneo.loc[broken, "municipality"].nunique())
print(sorted(ateneo.loc[broken, "municipality"].unique())[:8])
"PARAÃ‘AQUE".encode("cp1252").decode("utf-8")

rows: 363 | distinct names: 21
['BIÃ‘AN CITY', 'CITY OF BIÃ‘AN', 'CITY OF LAS PIÃ‘AS', 'CITY OF PARAÃ‘AQUE', 'DASMARIÃ‘AS CITY', 'DOÃ‘A REMEDIOS TRINIDAD', 'DUEÃ‘AS', 'LOS BAÃ‘OS']


'PARAÑAQUE'

In [13]:
import sys
sys.path.insert(0, str(PROJECT_ROOT))
from src.transform.clean_backfill import apply_town_maps

hf_clean = pd.read_parquet(STAGING / "hf_memberships_clean.parquet")
oh_clean = pd.read_parquet(STAGING / "openhalalan_winners_clean.parquet")
oh_ok = oh_clean[~oh_clean["town_shift_suspect"]]
known = set(zip(hf_clean["province_std"], hf_clean["locality_std"])) | set(zip(oh_ok["province_std"], oh_ok["town_std"]))

AT_PROV = {"NCR - MANILA": "NCR FIRST DISTRICT", "NCR - SECOND DISTRICT": "NCR SECOND DISTRICT",
           "NCR - THIRD DISTRICT": "NCR THIRD DISTRICT", "NCR - FOURTH DISTRICT": "NCR FOURTH DISTRICT",
           "TAWI-TAWI": "TAWI TAWI"}
ateneo["province_std"] = ateneo["province"].replace(AT_PROV)
ateneo["town_std"] = apply_town_maps(ateneo["municipality"], ateneo["province_std"])

has_town = ateneo["town_std"].notna()
is_known = pd.Series([pair in known for pair in zip(ateneo["province_std"], ateneo["town_std"])], index=ateneo.index)
unknown = ateneo[has_town & ~is_known]
print("rows with a town:", has_town.sum(), "| not a known town in that province:", len(unknown))
unknown["year"].value_counts().sort_index()

rows with a town: 173872 | not a known town in that province: 4481


year
1988    1200
1992     693
1995     438
1998     516
2013      20
2016     116
2019     263
2022    1235
Name: count, dtype: int64

In [14]:
unknown.groupby(["province_std", "municipality"]).size().sort_values(ascending=False).head(25)

province_std         municipality      
ZAMBOANGA DEL SUR    TUNGAWAN              40
                     BUUG                  40
                     ALICIA                40
                     DIPLAHAN              40
                     IMELDA                40
CAVITE               MENDEZ-NUNEZ          40
SORSOGON             BACON                 40
ZAMBOANGA DEL SUR    SIAY                  40
                     ROSELLER LIM          40
                     OLUTANGA              40
                     KABASALAN             40
                     MABUHAY               40
NCR FOURTH DISTRICT  PARAÃ‘AQUE            40
ZAMBOANGA DEL SUR    MALANGAS              40
                     TALUSAN               40
                     IPIL                  40
                     TITAY                 40
DAVAO DEL NORTE      MACO                  30
                     MAWAB                 30
                     NEW BATAAN            30
                     NABUNTURAN         

In [15]:
fn = ateneo["first_name"].fillna("")
fn.str.extract(r"\s(JR\.?|SR\.?|II|III|IV|V|VI|VII|VIII|111)$")[0].value_counts()

0
JR.     1683
SR.      277
III      174
II       103
JR        85
V         59
IV        31
VI         9
SR         7
111        6
VII        1
VIII       1
Name: count, dtype: int64

In [16]:
pct_space = fn.str.contains(" ").groupby(ateneo["year"]).mean().mul(100).round(1)
glued = fn.str.contains(r"[A-Z](?:JR|SR)\.?$", regex=True)
pd.DataFrame({"pct_with_space": pct_space, "suffix_glued": glued.groupby(ateneo["year"]).sum()})

,pct_with_space,suffix_glued
year,,
1987,8.8,0
1988,4.4,0
1992,3.5,0
1995,5.4,0
1998,6.1,0
2001,6.3,0
2004,7.6,0
2007,8.7,0
2010,10.8,0


In [17]:
odd_chars = r"[0-9;!?_|]"
bad_first = ateneo["first_name"].str.contains(odd_chars, na=False)
bad_last = ateneo["last_name"].str.contains(odd_chars, na=False)
print("first names:", bad_first.sum(), "| last names:", bad_last.sum())
print(ateneo.loc[bad_first, "first_name"].head(12).tolist())
print(ateneo.loc[bad_last, "last_name"].head(12).tolist())
ateneo.loc[bad_first | bad_last, "year"].value_counts().sort_index()

first names: 77 | last names: 14
['OKL?NO', 'HADJI SI8LY', 'I?IGO', 'EMED1NA', 'TI8URCIO', 'P?DRO', 'FILOMEN0', 'LUDEN1TO', 'AGUST?N', 'GA?I', 'HERI6ERTQ', 'DOM1NADOR']
['AL0UIZA', 'BAGAF0RO', 'BERMEJ0', 'CARIN0', 'CARIN0', 'CARIN0', 'DAMPIL;AG', 'DOMINGO!', 'ESTOP!A', 'G_ILAS', 'PELAGI0', 'RILL0']


year
1988    55
1992     4
2001     1
2004     3
2007     3
2010     6
2016     3
2019     7
2022     9
Name: count, dtype: int64

In [18]:
print("exact duplicate rows:", ateneo.duplicated().sum())
key = ["year", "position", "province", "municipality", "first_name", "last_name"]
dupes = ateneo[ateneo.duplicated(key, keep=False)].sort_values(key)
print("rows in a duplicate group (same name, office, town, year):", len(dupes))
dupes[key + ["party"]].head(10)

exact duplicate rows: 7
rows in a duplicate group (same name, office, town, year): 26


,year,position,province,municipality,first_name,last_name,party
67033,1987,"MEMBER, HOUSE OF REPRESENTATIVES",PANGASINAN,NaN,CONRADO,ESTRELLA,-
67034,1987,"MEMBER, HOUSE OF REPRESENTATIVES",PANGASINAN,NaN,CONRADO,ESTRELLA,-
67355,1995,COUNCILOR,BATANGAS,TINGLOY,ANASTACIA,EVANGELISTA,INDEPENDENT
67365,1995,COUNCILOR,BATANGAS,TINGLOY,ANASTACIA,EVANGELISTA,LABAN (LDP)
34255,1998,COUNCILOR,NUEVA ECIJA,GEN. TINIO,DANILO,BUSALPA,NO CERTIFICATE OF
34256,1998,COUNCILOR,NUEVA ECIJA,GEN. TINIO,DANILO,BUSALPA,PDR-LM
85088,1998,VICE MAYOR,PAMPANGA,MACABEBE,FIDEL,ISIP,NPC
85089,1998,VICE MAYOR,PAMPANGA,MACABEBE,FIDEL,ISIP,NPC
126435,2001,COUNCILOR,LEYTE,CALUBIAN,LEONARDO,PATES,LAKAS NUCD-UMDP
126436,2001,COUNCILOR,LEYTE,CALUBIAN,LEONARDO,PATES,LAKAS NUCD-UMDP


In [19]:
same_surname = ateneo.groupby(["year", "province", "last_name"])["last_name"].transform("size")
my_flag = (same_surname >= 2).map({True: "fat", False: "non-fat"})
agree = my_flag == ateneo["fat_dynasty_indicator"]
print("agreement:", round(agree.mean() * 100, 2), "%")
(~agree).groupby(ateneo["year"]).sum()

agreement: 99.92 %


year
1987      0
1988      0
1992      0
1995      0
1998      0
2001      0
2004      0
2007      0
2010      0
2013      0
2016      0
2019      5
2022    169
dtype: int64

In [20]:
odd_2022 = ateneo[~agree & ateneo["year"].eq(2022) & ateneo["fat_dynasty_indicator"].eq("fat")]
odd_2022[["province", "position", "first_name", "last_name"]].head(8)

,province,position,first_name,last_name
189641,BASILAN,GOVERNOR,JIM,HATAMAN-SALLIMAN
189654,LA UNION,GOVERNOR,RAFY,ORTEGA-DAVID
189777,SIQUIJOR,VICE GOVERNOR,MIMI,QUEZON-BROWN
189791,DAVAO DEL SUR,VICE GOVERNOR,AIAI,CAGAS-FERNANDEZ
189837,SULU,PROVINCIAL BOARD MEMBER,RAIHANA,ARBISON-ADAM
189924,LA UNION,PROVINCIAL BOARD MEMBER,JEN,MOSUELA-FERNANDEZ
189972,NUEVA VIZCAYA,PROVINCIAL BOARD MEMBER,ELMA,PINAO-AN-LEJAO
190003,BATAAN,PROVINCIAL BOARD MEMBER,IYA,ROQUE


In [21]:
ateneo["last_name"].str.contains("-", na=False).groupby(ateneo["year"]).sum()

year
1987      1
1988    107
1992     91
1995     80
1998     87
2001    107
2004    103
2007     98
2010    103
2013    184
2016    209
2019    238
2022    371
Name: last_name, dtype: int64

In [22]:
a = ateneo.groupby(["year", "position"]).size().rename("ateneo")
o = oh.groupby(["Year", "Position"]).size().rename("openhalalan")
o.index.names = ["year", "position"]
both = pd.concat([a, o], axis=1).astype("Int64")
both["difference"] = both["ateneo"] - both["openhalalan"]
both.loc[[2001, 2019, 2022]]

ateneo  openhalalan  difference
year position                                                         
2001 COUNCILOR                          12995        13180        -185
     GOVERNOR                              79           78           1
     MAYOR                               1585         1607         -22
     MEMBER, HOUSE OF REPRESENTATIVES     209          208           1
     PROVINCIAL BOARD MEMBER              722          732         -10
     VICE GOVERNOR                         77           78          -1
     VICE MAYOR                          1585         1607         -22
2019 COUNCILOR                          13427        13363          64
     GOVERNOR                              81           81           0
     MAYOR                               1626         1634          -8
     MEMBER, HOUSE OF REPRESENTATIVES     243          221          22
     PROVINCIAL BOARD MEMBER              781          768          13
     VICE GOVERNOR                         81           81           0
     VICE MAYOR                          1627         1633          -6
     SENATOR                             <NA>           12        <NA>
2022 COUNCILOR                          13499        13376         123
     GOVERNOR                              81           81           0
     MAYOR                               1631         1634          -3
     MEMBER, HOUSE OF REPRESENTATIVES     253          228          25
     PROVINCIAL BOARD MEMBER              782          770          12
     VICE GOVERNOR                         81           81           0
     VICE MAYOR                          1631         1634          -3
     PRESIDENT                           <NA>            1        <NA>
     SENATOR                             <NA>           12        <NA>
     VICE PRESIDENT                      <NA>            1        <NA>

In [23]:
print("rows from 1987-1998:", ateneo["year"].le(1998).sum())

mv = ateneo[ateneo["position"].isin(["MAYOR", "VICE MAYOR"]) & ~ateneo["year"].isin([2013, 2016])]
per_town = mv.groupby(["year", "province", "municipality", "position"]).size().unstack(fill_value=0)
incomplete = per_town[(per_town["MAYOR"] != 1) | (per_town["VICE MAYOR"] != 1)]
print("towns without exactly 1 mayor and 1 vice mayor:", incomplete.groupby(level="year").size().to_dict())

key = ["year", "position", "province", "municipality", "first_name", "last_name"]
copies = ateneo.duplicated(key)
in_group = ateneo.duplicated(key, keep=False)
n_after = ateneo[~copies].groupby(["year", "province", "last_name"])["last_name"].transform("size")
print("officials flagged fat only because of their own duplicate:", (n_after[in_group[~copies]] == 1).sum())

rows from 1987-1998: 68289
towns without exactly 1 mayor and 1 vice mayor: {1988: 6, 1992: 5, 1995: 2, 1998: 9, 2001: 6, 2004: 1, 2007: 5, 2010: 39, 2019: 1}
officials flagged fat only because of their own duplicate: 8


### Summary: ateneo_politicians (raw)

**What it is:** The Ateneo Policy Center Philippine Political Dynasties Dataset (2022 update), sheet "Data - Politicians". One row per elected official per election. 207,599 rows, 10 columns: `first_name`, `last_name`, `party`, `region`, `province`, `municipality`, `position`, `year`, `PSGC_Province`, `fat_dynasty_indicator`. No middle names.

**Coverage**
- 13 election years: 1987 (House members only, 193 rows), 1988 (local officials only), then every 3 years from 1992 to 2022 (House and local together). About 17,000 rows per election.
- Positions: governor, vice governor, provincial board member, mayor, vice mayor, councilor, and district House members. No senators, president, vice president, or party-list (by design, per the dataset's disclaimer).
- A few towns lack a mayor or vice mayor record (e.g., 39 towns in 2010, 9 in 1998).

**Relationship with HF and OpenHalalan**
- **HF's 2004–2016 memberships are Ateneo's 2004–2016 rows:** identical row counts every year, identical missing towns (15,842 in 2013, 6,180 in 2016), and identical surnames except four typos HF corrected (CARIN0, BERMEJ0, SILVERI0, DAMPIL;AG). HF adds person IDs, Title Case names, and separate suffixes. HF is therefore not an independent source for these years, and Ateneo cannot fill the 2013/2016 town gaps.
- **New coverage:** 1987–1998 (68,289 rows; no other source has local officials for these years). 2001 and 2019–2022 overlap OpenHalalan.
- **Overlap with OpenHalalan:** in 2001, OpenHalalan is more complete (Ateneo has 22 fewer mayors and vice mayors and 185 fewer councilors). In 2019 and 2022, Ateneo has more councilors (+64, +123), House members (+22, +25), and board members (+13, +12); OpenHalalan has slightly more mayors (+8, +3) and the national offices. Neither source is complete.

**Provinces**
- 85 province names, each with exactly one PSGC code.
- 5 names differ from the standard: NCR - MANILA, NCR - SECOND/THIRD/FOURTH DISTRICT, TAWI-TAWI.
- New provinces appear only from their creation: Sarangani (1992), Biliran and Guimaras (1995), Davao de Oro (1998), Zamboanga Sibugay (2001), Dinagat Islands (2007), Davao Occidental (2016). Before that, their towns are listed under the parent province (e.g., Ipil: Zamboanga del Sur until 1998, Zamboanga Sibugay from 2001). Linking people within one province therefore splits careers and families at the change (e.g., Rey Andre Olegario, mayor of Ipil 1998–2004, followed by Anamel and Inday Amy Olegario).
- 58 NCR rows are filed under the wrong district: Pasig (42), Pateros (12), San Juan (3), Navotas (1).

**Towns (`municipality`)**
- Blank in 33,727 rows: most provincial positions, plus the same town-level gaps as HF (15,842 in 2013, 6,180 in 2016).
- 2022: all 1,197 provincial officials have their position title in the town column (e.g., "MEMBER, SANGGUNIANG PANLALAWIGAN").
- 2019: House members are listed with one town of their district, which is not the office's location.
- Encoding error: Ñ appears as "Ã‘" in 363 rows (21 town names). Reversible with `text.encode("cp1252").decode("utf-8")`.
- After our existing town cleaning, 4,481 rows do not match a known (province, town) pair, mostly in 1988–1998 and 2022. Causes: towns now in a new province, position titles, encoding, abbreviations (GEN. S.K. PENDATUN), renamed towns (MUNOZ, ESPIRITU), towns that no longer exist (BACON, merged into Sorsogon City), Manila districts used as towns (TONDO, BINONDO), and typos.

**Names**
- About 2,400 first names end with a suffix (JR. 1,683; SR. 277; III 174; II 103; JR 85; IV 31; VI 9; SR 7; VII 1; VIII 1). "111" (6 rows) is III. A final "V" (59 rows) is ambiguous: the Fifth or a middle initial.
- 2019 lost every space in first names: 0.1% contain a space, vs 4–23% in other years ("MARYRUTH", "SALICJR."; 677 suffixes glued to the name).
- 91 names (77 first, 14 last) contain scanning errors: 0 for O, 1 for I, 8 or 6 for B, ? for an unreadable letter. 55 of them are from 1988.
- 1 row has no first name.
- 13 extra copies of rows (7 identical; the rest differ only in party, e.g., the incomplete "NO CERTIFICATE OF"). HF has the same duplicates for 2004–2016.
- Hyphenated surnames grow from about 100 per election (1988–2010) to 371 in 2022.

**`fat_dynasty_indicator`** (52,932 rows "fat", 25.5%)
- Reproduced exactly for 1987–2016 by one rule: at least one other official (any position, including House) with the same surname, in the same province, in the same election year. Overall agreement 99.92%.
- 2019 (5 rows) and 2022 (169 rows) differ. In 2022, hyphenated surnames count for both families (e.g., ORTEGA-DAVID with ORTEGA).
- It is a surname rule, not confirmed kinship, so it is an upper bound on dynasties. Duplicate rows make 8 officials "fat" through their own copy.

**Usage:** the file's Disclaimer sheet states it "may not be further copied, distributed or publicly displayed without the express written permission of the Ateneo School of Government." Keep the raw file out of the repository and cite it as "The Ateneo Policy Center Philippine Political Dynasties Dataset."

## 8. ateneo_provinces (raw)

In [24]:
shares = pd.read_parquet(RAW / "ateneo_provinces_raw.parquet")
print(shares.shape)
shares.info()
shares.head()

(81, 12)
<class 'pandas.DataFrame'>
RangeIndex: 81 entries, 0 to 80
Data columns (total 12 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   province         81 non-null     str    
 1   fatdynshare1992  75 non-null     float64
 2   fatdynshare1995  77 non-null     float64
 3   fatdynshare1998  78 non-null     float64
 4   fatdynshare2001  79 non-null     float64
 5   fatdynshare2004  79 non-null     float64
 6   fatdynshare2007  80 non-null     float64
 7   fatdynshare2010  80 non-null     float64
 8   fatdynshare2013  80 non-null     float64
 9   fatdynshare2016  81 non-null     float64
 10  fatdynshare2019  81 non-null     float64
 11  fatdynshare2022  81 non-null     float64
dtypes: float64(11), str(1)
memory usage: 8.5 KB


,province,fatdynshare1992,fatdynshare1995,fatdynshare1998,fatdynshare2001,fatdynshare2004,fatdynshare2007,fatdynshare2010,fatdynshare2013,fatdynshare2016,fatdynshare2019,fatdynshare2022
0,ABRA,25.622776,28.825623,27.402136,32.384342,28.782288,32.103321,32.740215,35.587189,31.672598,33.096085,37.722420
1,AGUSAN DEL NORTE,12.903226,13.432836,17.164179,18.656717,15.441176,16.129032,16.911764,18.400000,24.264706,22.794117,22.794117
2,AGUSAN DEL SUR,11.258278,17.647058,19.607843,20.915033,16.339869,20.325203,21.794872,22.435898,23.717949,24.358974,28.846153
3,AKLAN,21.111111,32.596684,29.281769,29.281769,28.961748,23.121387,31.034483,34.090908,30.434782,30.434782,34.782608
4,ALBAY,14.213198,19.251337,13.705584,16.842106,22.388060,14.427860,19.900497,18.000000,20.603016,18.905472,27.860697


In [25]:
shares.describe().round(1)

,fatdynshare1992,fatdynshare1995,fatdynshare1998,fatdynshare2001,fatdynshare2004,fatdynshare2007,fatdynshare2010,fatdynshare2013,fatdynshare2016,fatdynshare2019,fatdynshare2022
count,75.0,77.0,78.0,79.0,79.0,80.0,80.0,80.0,81.0,81.0,81.0
mean,19.6,21.8,21.1,21.8,22.2,22.3,25.0,26.3,26.9,27.6,28.7
std,10.0,9.7,9.8,9.1,9.1,8.3,8.9,9.2,9.0,8.8,8.9
min,0.0,1.8,0.0,1.8,2.7,1.8,1.8,1.8,3.6,6.3,3.6
25%,11.5,14.9,14.7,15.3,16.3,17.3,18.9,20.9,21.3,22.0,22.8
50%,20.0,20.9,20.8,21.5,22.0,22.1,24.6,25.3,26.9,28.5,28.4
75%,25.9,28.4,27.5,26.6,27.2,26.5,29.3,32.0,31.7,32.2,33.1
max,44.5,45.0,48.8,48.5,47.7,44.4,51.7,47.1,49.2,50.5,52.8


In [26]:
long = shares.melt(id_vars="province", var_name="column", value_name="share")
long["year"] = long["column"].str[-4:].astype(int)

mine = ateneo.groupby(["province", "year"])["fat_dynasty_indicator"].apply(lambda s: (s == "fat").mean() * 100).rename("my_share").reset_index()
check = long.merge(mine, on=["province", "year"], how="left")
check["diff"] = (check["share"] - check["my_share"]).abs()
print("values compared:", check["share"].notna().sum(), "| max difference:", round(check["diff"].max(), 4))

values compared: 871 | max difference: 0.0


In [27]:
missing = long[long["share"].isna()]
missing.groupby("province")["year"].apply(list)

province
BILIRAN                                                        [1992]
DAVAO DE ORO                                             [1992, 1995]
DAVAO OCCIDENTAL     [1992, 1995, 1998, 2001, 2004, 2007, 2010, 2013]
DINAGAT ISLANDS                        [1992, 1995, 1998, 2001, 2004]
GUIMARAS                                                       [1992]
ZAMBOANGA SIBUGAY                                  [1992, 1995, 1998]
Name: year, dtype: object

In [28]:
sorted(set(ateneo["province"]) - set(shares["province"]))

['NCR - FOURTH DISTRICT',
 'NCR - MANILA',
 'NCR - SECOND DISTRICT',
 'NCR - THIRD DISTRICT']

### Summary: ateneo_provinces (raw)

**What it is:** Sheet "Data - Province" of the Ateneo dataset. One row per province, one column per election (1992–2022): the **fat dynasty share**, i.e. the percentage of the province's elected officials flagged "fat" in that election. 81 rows, 12 columns.

**Quality**
- Recomputed exactly from the politicians sheet (871 values, max difference 0.0): share = % of all officials in the province and year (all positions, including House) with `fat_dynasty_indicator == "fat"`. The sheet is therefore derived data, not an independent source, and inherits the flag's limits (surname rule, duplicates, 2022 method change).
- Blank shares (20 values) belong to provinces that did not exist yet: Biliran and Guimaras (1992), Davao de Oro (1992–1995), Zamboanga Sibugay (1992–1998), Dinagat Islands (1992–2004), Davao Occidental (1992–2013). Each blank ends one election before the province first appears in the politicians sheet. Blanks must stay blank, not 0.
- Not included: the 4 NCR districts (Ateneo reports shares for provinces only) and the 1987/1988 term (the sheet starts in 1992).

**Descriptive statistics**
- Average share across provinces rises from 19.6% (1992) to 28.7% (2022); median from 20.0% to 28.4%; standard deviation stays around 9 points.
- Highest in 2022: Maguindanao (52.8%). Bulacan is highest in 1992 and 1998, which reflects very common surnames as much as dynasties, a direct effect of the surname rule.

**Use in the project:** a published benchmark for comparing task 8's town and province stronghold measures (which use kinship links instead of shared surnames).

## 9. Uniform profile of all sources

In [29]:
import sys
sys.path.insert(0, str(PROJECT_ROOT))
from src.utils.profiling import profile_table, source_summary

sources = {
    "hf_persons": pd.read_parquet(RAW / "hf_persons.parquet"),
    "hf_memberships": pd.read_parquet(RAW / "hf_memberships.parquet"),
    "openhalalan_winners": pd.read_csv(RAW / "openhalalan_winners.csv"),
    "psa_poverty (staging)": pd.read_csv(STAGING / "psa_poverty.csv"),
    "roster_legislators (staging)": pd.read_csv(STAGING / "roster_legislators.csv"),
    "ateneo_politicians": pd.read_parquet(RAW / "ateneo_politicians_raw.parquet"),
    "ateneo_provinces": pd.read_parquet(RAW / "ateneo_provinces_raw.parquet"),
}
source_summary(sources)

,rows,columns,duplicate_rows,missing_cells_pct
hf_persons,45424,4,0,24.0
hf_memberships,86234,8,0,4.0
openhalalan_winners,157333,14,0,10.6
psa_poverty (staging),320,13,0,0.8
roster_legislators (staging),5118,15,0,9.6
ateneo_politicians,207599,10,7,1.6
ateneo_provinces,81,12,0,2.1


In [30]:
for name, df in sources.items():
    print(f"\n=== {name} ===")
    display(profile_table(df))


=== hf_persons ===


,dtype,non_null,missing_pct,unique,top_value,top_pct
id,str,45424,0.0,45424,01K7QAQB1SY5XFF0N3Y7S168NF,0.0
first_name,str,45424,0.0,13122,Jose,1.0
last_name,str,45424,0.0,18765,Tan,0.4
name_suffix,str,1794,96.1,13,Jr.,71.7



=== hf_memberships ===


,dtype,non_null,missing_pct,unique,top_value,top_pct,min,max,mean
id,str,86234,0.0,86234,01K7QAQB1SY5XFF0N3Y7S168NF_m0,0.0,NaN,NaN,NaN
person_id,str,86234,0.0,45424,01K7QAK67EFRCFNPAVQ5WTTPTZ,0.0,NaN,NaN,NaN
party,str,86234,0.0,368,LP,21.6,NaN,NaN,NaN
region,str,86234,0.0,17,REGION VIII,8.6,NaN,NaN,NaN
province,str,86234,0.0,85,CEBU,3.3,NaN,NaN,NaN
locality,str,58594,32.1,1463,SAN JOSE,0.5,NaN,NaN,NaN
position,str,86234,0.0,7,COUNCILOR,75.1,NaN,NaN,NaN
year,int32,86234,0.0,5,2016,20.4,2004.0,2016.0,2010.03



=== openhalalan_winners ===


...

,dtype,non_null,missing_pct,unique,top_value,top_pct,min,max,mean
name,str,5118,0.0,2520,"FESTIN, LEONARDO F.",0.2,NaN,NaN,NaN
surname,str,5118,0.0,1485,GARCIA,0.7,NaN,NaN,NaN
region_province,str,5054,1.3,306,Party-List,5.4,NaN,NaN,NaN
district,str,4448,13.1,131,First,25.3,NaN,NaN,NaN
period_clean,str,5117,0.0,64,17th Congress,5.9,NaN,NaN,NaN
congress_number,float64,3217,37.1,17,17.0,9.4,1.0,17.0,10.92
start_year,float64,5115,0.1,78,2016.0,5.8,1907.0,2019.0,1974.47
end_year,float64,5115,0.1,88,2016.0,5.7,1908.0,2019.0,1977.32
years_raw,str,5115,0.1,182,2013-2016,5.5,NaN,NaN,NaN
years_corrected,bool,5118,0.0,2,False,99.9,NaN,NaN,NaN



=== ateneo_politicians ===


,dtype,non_null,missing_pct,unique,top_value,top_pct,min,max,mean
first_name,str,207598,0.0,28988,JOSE,1.1,NaN,NaN,NaN
last_name,str,207599,0.0,30429,TAN,0.4,NaN,NaN,NaN
party,str,207591,0.0,1956,LAKAS NUCD-UMDP,11.6,NaN,NaN,NaN
region,str,207599,0.0,19,REGION IV-A,8.7,NaN,NaN,NaN
province,str,207599,0.0,85,CEBU,3.3,NaN,NaN,NaN
municipality,str,173872,16.2,2006,SAN JOSE,0.5,NaN,NaN,NaN
position,str,207599,0.0,7,COUNCILOR,75.3,NaN,NaN,NaN
year,int64,207599,0.0,13,2022,8.7,1987.0,2022.0,2005.58
PSGC_Province,str,207599,0.0,85,PH072200000,3.3,NaN,NaN,NaN
fat_dynasty_indicator,str,207599,0.0,2,non-fat,74.5,NaN,NaN,NaN



=== ateneo_provinces ===


,dtype,non_null,missing_pct,unique,top_value,top_pct,min,max,mean
province,str,81,0.0,81,ABRA,1.2,NaN,NaN,NaN
fatdynshare1992,float64,75,7.4,74,10.169492,2.7,0.00,44.53,19.61
fatdynshare1995,float64,77,4.9,74,11.428572,2.6,1.83,44.96,21.78
fatdynshare1998,float64,78,3.7,77,14.754098,2.6,0.00,48.78,21.10
fatdynshare2001,float64,79,2.5,78,11.290322,2.5,1.80,48.46,21.84
fatdynshare2004,float64,79,2.5,78,16.923077,2.5,2.70,47.72,22.22
fatdynshare2007,float64,80,1.2,77,23.121387,2.5,1.83,44.39,22.29
fatdynshare2010,float64,80,1.2,80,32.740215,1.2,1.80,51.67,25.01
fatdynshare2013,float64,80,1.2,79,29.629629,2.5,1.80,47.11,26.25
fatdynshare2016,float64,81,0.0,81,31.672598,1.2,3.60,49.21,26.91


### Summary: uniform profile of all sources

| Source | Rows | Columns | Duplicate rows | Missing cells | Years |
|---|---|---|---|---|---|
| hf_persons | 45,424 | 4 | 0 | 24.0% | n/a |
| hf_memberships | 86,234 | 8 | 0 | 4.0% | 2004–2016 |
| openhalalan_winners | 157,333 | 14 | 0 | 10.6% | 2001–2025 |
| psa_poverty (staging) | 320 | 13 | 0 | 0.8% | 2018–2023 |
| roster_legislators (staging) | 5,118 | 15 | 0 | 9.6% | 1907–2019 |
| ateneo_politicians | 207,599 | 10 | 7 | 1.6% | 1987–2022 |
| ateneo_provinces | 81 | 12 | 0 | 2.1% | 1992–2022 |

- Together the sources cover House members from 1907 and local officials from 1987, up to 2025.
- Most missing values are structural, not data loss: `name_suffix` (96.1%, most people have no suffix), OpenHalalan `Title` (99.3%), roster `notes` (92.4%), `congress_number` (37.1%, legislatures before Congress numbering). The real gaps are in the town columns (HF `locality` 32.1%, Ateneo `municipality` 16.2%, OpenHalalan `City` 8.6%), analyzed in sections 2, 3, and 7.
- Party names are not standardized in any source: 1,956 distinct values in Ateneo, 421 in OpenHalalan, 368 in HF. The project does not use party in its analysis, so it is kept as reported (limitation).
- Value ranges are valid: poverty incidence 0.3–75.3%, fat dynasty shares 0–52.85%, and all years within each source's coverage. PSA's `cv` reaches 49.2, so some provincial poverty estimates are very imprecise.